# Preprocess SuStaIn Asymmetric Biomarkers (Clean CN Filtered Baseline)

This notebook mirrors `preprocess_sustain_asymmetric.ipynb` exactly, but with one key refinement:
- **Filter CN Controls:** Out of the 186 total CN subjects, it identifies and isolates the **149 clean/normal CN controls** (excluding the 37 CN subjects who exhibited focal or global atrophy $Z > 2.0$).
- **Refitted Baseline:** Covariate regression (`Volume ~ ICV + Age + Sex`) and reference statistics ($\text{Mean}_{\text{clean CN}}$ and $\text{Std}_{\text{clean CN}}$) are calculated using **only these 149 clean CN controls**.
- **Output:** Exports clean Z-scored dataset ready for SuStaIn modeling to `csvs/adni_mri_sustain_prepared_asymmetric_filter_cn.csv`.

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.linear_model import LinearRegression

# Paths
raw_data_path = "/Users/khoale/Downloads/Alzheimer_Code/csvs/adni_mri_ucsf_merged.csv"
demographics_path = "/Users/khoale/Downloads/Alzheimer_Code/TABLE_DATA_ADNI/MRI_T1_3D_ADNI1_2_BL_PRE_4_24_2026.csv"
output_path = "/Users/khoale/Downloads/Alzheimer_Code/csvs/adni_mri_sustain_prepared_asymmetric_filter_cn.csv"

# Load raw dataset
df = pd.read_csv(raw_data_path)
print(f"Loaded raw data: {df.shape[0]} subjects, {df.shape[1]} columns")

# Load demographics
dem_df = pd.read_csv(demographics_path)[['Image Data ID', 'Age', 'Sex']]

# Merge using Image ID as key
df = df.merge(dem_df, left_on='MRI_ImageID', right_on='Image Data ID', how='left')
print(f"Merged demographics successfully by Image ID.")
print(f"  Missing values after merge - Age: {df['Age'].isna().sum()}, Sex: {df['Sex'].isna().sum()}")

In [ ]:
# Map columns belonging to each of our 24 asymmetric biomarkers
biomarker_mapping = {
    # --- LEFT HEMISPHERE ---
    "L_Frontal": ["ST15CV", "ST25CV", "ST36CV", "ST39CV", "ST43CV", "ST45CV", "ST46CV", "ST47CV", "ST51CV", "ST55CV", "ST56CV"],
    "L_Temporal": ["ST13CV", "ST24CV", "ST26CV", "ST32CV", "ST40CV", "ST44CV", "ST58CV", "ST60CV", "ST62CV"],
    "L_Parietal": ["ST31CV", "ST49CV", "ST52CV", "ST57CV", "ST59CV"],
    "L_Occipital": ["ST23CV", "ST35CV", "ST38CV", "ST48CV"],
    "L_Cingulate": ["ST14CV", "ST34CV", "ST50CV", "ST54CV"],
    "L_Insula": ["ST129CV"],
    "L_Hippocampus": ["ST29SV"],
    "L_Amygdala": ["ST12SV"],
    "L_Caudate": ["ST16SV"],
    "L_Pallidum": ["ST42SV"],
    "L_Putamen": ["ST53SV"],
    "L_Accumbens": ["ST11SV"],

    # --- RIGHT HEMISPHERE ---
    "R_Frontal": ["ST74CV", "ST84CV", "ST95CV", "ST98CV", "ST102CV", "ST104CV", "ST105CV", "ST106CV", "ST110CV", "ST115CV"],
    "R_Temporal": ["ST72CV", "ST83CV", "ST85CV", "ST91CV", "ST99CV", "ST103CV", "ST117CV", "ST119CV", "ST121CV"],
    "R_Parietal": ["ST90CV", "ST108CV", "ST111CV", "ST116CV", "ST118CV"],
    "R_Occipital": ["ST82CV", "ST94CV", "ST97CV", "ST107CV"],
    "R_Cingulate": ["ST73CV", "ST93CV", "ST109CV", "ST113CV"],
    "R_Insula": ["ST130CV"],
    "R_Hippocampus": ["ST88SV"],
    "R_Amygdala": ["ST71SV"],
    "R_Caudate": ["ST75SV"],
    "R_Pallidum": ["ST101SV"],
    "R_Putamen": ["ST112SV"],
    "R_Accumbens": ["ST70SV"]
}

In [ ]:
# 1. Force all ROI columns and ICV (ST10CV) to numeric values
all_roi_cols = [col for cols in biomarker_mapping.values() for col in cols]
cols_to_convert = all_roi_cols + ["ST10CV"]

for col in cols_to_convert:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 2. Drop rows where Intracranial Volume (ICV) is missing
df = df.dropna(subset=["ST10CV"]).copy()

# 3. Sum sub-regions for Left and Right separately, keeping covariates
aggregated_df = pd.DataFrame()
aggregated_df["PTID"] = df["PTID"]
aggregated_df["Label"] = df["Label"]
aggregated_df["ICV"] = df["ST10CV"]
aggregated_df["AGE"] = df["Age"]
aggregated_df["Sex_Code"] = df["Sex"].map({'Male': 1, 'Female': 0, 'M': 1, 'F': 0})

for region, cols in biomarker_mapping.items():
    aggregated_df[region] = df[cols].sum(axis=1, min_count=1)

print(f"Aggregated asymmetric data shape: {aggregated_df.shape}")

In [ ]:
# STEP 4: IDENTIFY THE 149 CLEAN CN CONTROLS
# First pass regression & Z-scoring on all 186 CN controls to flag abnormal CN subjects (Z > 2.0)
regions = list(biomarker_mapping.keys())
all_cn = aggregated_df[aggregated_df["Label"] == "CN"].copy()

temp_norm = aggregated_df.copy()
for region in regions:
    clean_cn = all_cn.dropna(subset=[region, "ICV", "AGE", "Sex_Code"])
    model = LinearRegression().fit(clean_cn[["ICV", "AGE", "Sex_Code"]].values, clean_cn[region].values)
    residuals = aggregated_df[region].values - model.predict(aggregated_df[["ICV", "AGE", "Sex_Code"]].values)
    temp_norm[region] = residuals + clean_cn[region].mean()

temp_cn = temp_norm[temp_norm["Label"] == "CN"].copy()
temp_z = pd.DataFrame()
for region in regions:
    mean_val = temp_cn[region].mean()
    std_val = temp_cn[region].std() + 1e-9
    temp_z[region] = (mean_val - temp_cn[region]) / std_val

# Count regions with Z > 2.0 for each CN patient
abnormal_roi_count = (temp_z > 2.0).sum(axis=1)
clean_cn_ptids = all_cn.loc[abnormal_roi_count == 0, "PTID"].values

print(f"Original CN subjects: {len(all_cn)}")
print(f"Clean CN subjects retained: {len(clean_cn_ptids)} (Excluded {len(all_cn) - len(clean_cn_ptids)} abnormal CN subjects)")

In [ ]:
# STEP 5: REFIT COVARIATE REGRESSION USING ONLY THE 149 CLEAN CN CONTROLS
normalized_df = aggregated_df.copy()
clean_cn_controls = aggregated_df[aggregated_df["PTID"].isin(clean_cn_ptids)].copy()

print(f"Refitting ICV + Age + Sex regression using {len(clean_cn_controls)} clean CN controls...")

for region in regions:
    clean_cn = clean_cn_controls.dropna(subset=[region, "ICV", "AGE", "Sex_Code"])
    
    # Fit regression model on clean CN only
    X_clean = clean_cn[["ICV", "AGE", "Sex_Code"]].values
    y_clean = clean_cn[region].values
    
    model = LinearRegression()
    model.fit(X_clean, y_clean)
    
    # Calculate residuals for ALL subjects in dataset
    X_all = aggregated_df[["ICV", "AGE", "Sex_Code"]].values
    y_all = aggregated_df[region].values
    
    predicted_all = model.predict(X_all)
    residuals = y_all - predicted_all
    
    # Add back mean volume of CLEAN CN controls
    mean_volume_clean_cn = y_clean.mean()
    normalized_df[region] = residuals + mean_volume_clean_cn

print("Covariates correction using Clean CN reference complete.")

In [ ]:
# STEP 6: CALCULATE FINAL Z-SCORES RELATIVE TO CLEAN CN CONTROLS
sustain_ready_df = pd.DataFrame()
sustain_ready_df["PTID"] = normalized_df["PTID"]
sustain_ready_df["Label"] = normalized_df["Label"]

# Reference stats computed ONLY on clean CN controls
clean_cn_normalized = normalized_df[normalized_df["PTID"].isin(clean_cn_ptids)]

for region in regions:
    mean_clean = clean_cn_normalized[region].mean()
    std_clean = clean_cn_normalized[region].std() + 1e-9
    
    # Inverted formula for atrophy: Z = (mean_clean - actual) / std_clean
    sustain_ready_df[region] = (mean_clean - normalized_df[region]) / std_clean

# 1. Fill NaNs with 0
sustain_ready_df[regions] = sustain_ready_df[regions].fillna(0)

# 2. Clamp negative Z-scores to 0 (SuStaIn ready format)
sustain_ready_df[regions] = sustain_ready_df[regions].clip(lower=0)

print("Preprocessing complete! Preview of final asymmetric z-scores (Clean CN Baseline):")
sustain_ready_df[regions].describe()

In [ ]:
# STEP 7: EXPORT PREPARED DATASET
sustain_ready_df.to_csv(output_path, index=False)
print(f"Clean CN filtered dataset exported successfully to:\n{output_path}")